In [1]:
import torch
import torch.nn as nn
import torch.optim as optimizer
from torch.utils.data import Dataset, DataLoader

# PyTorch Geometric imports
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool

# For debugging or general utilities (optional)
import numpy as np  # If you need to manipulate arrays
import random  # For reproducibility or shuffling sequences

In [51]:
class GATv2SequenceModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, edge_dim, out_channels, heads=1):
        super(GATv2SequenceModel, self).__init__()
        
        # GATv2 Layers
        self.gat1 = GATv2Conv(in_channels, hidden_channels, heads=heads, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, edge_dim=edge_dim)
        
        # GRU for Sequence Processing
        self.gru = torch.nn.GRU(hidden_channels * heads, hidden_channels, batch_first=True,bidirectional=True)
        
        # Fully Connected Layer for Predictions
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data_list):
        graph_embeddings = []
    
        for data in data_list:
            x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
            
            # GATv2 Layers
            x = self.gat1(x, edge_index, edge_attr).relu()
            x = self.gat2(x, edge_index, edge_attr).relu()
            
            # Aggregate node features into graph embeddings
            graph_embedding = global_mean_pool(x, data.batch)  # [batch_size, hidden_channels]
            graph_embeddings.append(graph_embedding)
        
        # Stack graph embeddings into a sequence tensor
        graph_sequence = torch.stack(graph_embeddings, dim=1)  # [batch_size, seq_len, hidden_channels]
        
        # Pass through GRU
        _, h_n = self.gru(graph_sequence)
        
        # Final output layer
        out = self.fc(h_n[-1])  # Use the final GRU state
        return out

In [3]:

def create_graph_array(train_d: dict) -> list:
    graph_list = []
    for k, v in train_d.items():
        nodes = torch.tensor([], dtype=torch.float)
        edge_indexes = torch.tensor([[], []], dtype=torch.long)
        edge_features = torch.tensor([], dtype=torch.float)
        k = str(k)
        v = dict(v)
        ego = v.pop("ego_vehicle")
        ego_pos = ego["position"]
        ego_wayp = ego["waypoint_location"]
        ego_node = torch.tensor(
            [
                [
                    ego_pos["x"],
                    ego_pos["y"],
                    ego_pos["z"],
                    ego_wayp["x"],
                    ego_wayp["y"],
                    ego_wayp["z"],
                    ego["speed"],
                    0,
                    0,
                    0,
                ]
            ],
            dtype=torch.float,
        )
        nodes = torch.cat((nodes, ego_node), dim=0)
        index = 1
        for key, value in v.items():
            exo_pos = value["Position"]
            exo_rotation = value["Rotation"]
            exo_velocity = value["Velocity"]
            exo_rel_pos = value["relative_position"][0]
            exo_rel_dir = value["relative_direction"]
            exo_node = torch.tensor(
                [
                    [
                        exo_pos["x"],
                        exo_pos["y"],
                        exo_pos["z"],
                        exo_rotation["pitch"],
                        exo_rotation["yaw"],
                        exo_rotation["roll"],
                        value["Speed"],
                        exo_velocity["x"],
                        exo_velocity["y"],
                        exo_velocity["z"],
                    ]
                ],
                dtype=torch.float,
            )
            nodes = torch.cat((nodes, exo_node), dim=0)
            e_index = torch.tensor([[0, index], [index, 0]], dtype=torch.long)
            edge_indexes = torch.cat((edge_indexes, e_index), dim=1)
            e_attribute = torch.tensor(
                [
                    [
                        exo_rel_pos["x"],
                        exo_rel_pos["y"],
                        exo_rel_pos["z"],
                        exo_rel_dir["x"],
                        exo_rel_dir["y"],
                        exo_rel_dir["z"],
                    ]
                ],
                dtype=torch.float,
            )
            edge_features = torch.cat((edge_features, e_attribute), dim=0)
            edge_features = torch.cat((edge_features, e_attribute), dim=0)

            index += 1
            graph = Data(x=nodes, edge_index=edge_indexes, edge_attr=edge_features)
            graph_list.append(graph)

    return graph_list

In [37]:
import json
import networkx as nx
from torch_geometric.utils import to_networkx
with open("formatted.json","r") as f:
    test_set=json.load(f)
a=create_graph_array(test_set)
# b=to_networkx(a[0])
# nx.draw(b)

In [18]:
sequence_length = 20
sequences = [a[i:i + sequence_length] for i in range(0, len(a), sequence_length)]
sequences[0]
dataset=[]
for i in sequences:
    dataset.append((i,random.choice([0,1,2,3,4])))

In [20]:
def collate_fn(batch):
    """
    Custom collate function for handling sequences of graphs.
    Input:
        batch: List of tuples (sequence of graphs, label)
    Output:
        sequences: List of sequences (each sequence is a list of graphs)
        labels: Tensor of labels
    """
    sequences, labels = zip(*batch)
    return sequences, torch.tensor(labels, dtype=torch.long)


train_loader = DataLoader(
    dataset=dataset, batch_size=1, shuffle=True, collate_fn=collate_fn
)

In [49]:
in_channels = 10
hidden_channels = 16
out_channels = 5
edge_dim = 6
heads = 1
num_epochs = 2
batch_size = 1
learning_rate = 0.001



model = GATv2SequenceModel(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    edge_dim=edge_dim,
    out_channels=out_channels,
    heads=heads,
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for sequences, labels in train_loader:
        # Move data to GPU if available
        sequences=sequences[0]
        for i in sequences:
            i.to("cuda")
        labels = labels.to("cuda")
        outputs = model(sequences)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # Validation step (optional)
    # model.eval()
    # correct = 0
    # total = 0
    # with torch.no_grad():
    #     for sequences, labels in val_loader:
    #         sequences = [[graph.to("cuda") for graph in seq] for seq in sequences]
    #         labels = labels.to("cuda")

    #         outputs = model(sequences)
    #         _, predicted = torch.max(outputs, dim=1)
    #         correct += (predicted == labels).sum().item
    #         total += labels.size(0)

    # accuracy = correct / total * 100
    # print(f"Validation Accuracy: {accuracy:.2f}%")

tensor([[27.2365,  7.3870,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  4.9699, 16.5988,  0.0000, 52.4365, 25.3491,  0.0000],
        [17.0895,  2.4623,  1.1141,  0.0000,  0.0000,  5.6440,  6.1354,  3.7009,
          0.0000,  0.9968,  1.6566,  5.5329,  2.8164, 17.5879, 13.2330,  0.0000],
        [17.5701,  1.9748,  0.8356,  0.0000,  0.0000,  4.2330,  4.6015,  2.7756,
          0.0000,  0.7476,  0.9468,  8.4979,  2.1123, 22.7723, 15.0311,  0.0000],
        [11.6529,  2.9959,  2.4988,  0.0000,  0.0000,  2.2924,  1.8406,  1.1103,
          2.2115,  0.2991,  0.9751,  6.7983,  2.1447, 18.6482, 10.5899,  0.0000],
        [25.4288,  5.0866,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  2.6546, 17.7198,  0.0000, 47.6707, 23.9549,  0.0000],
        [16.6439,  1.6955,  0.3080,  0.0000,  0.0000,  5.2146,  5.7427,  4.6359,
          0.0000,  0.2505,  0.8849,  5.9066,  2.5443, 16.7538, 12.9027,  0.0000],
        [16.7987,  1.7

RuntimeError: shape '[20, 3, 16]' is invalid for input of size 320

In [28]:
torch.save(model.state_dict(),"trained.pth")